<a href="https://colab.research.google.com/github/UW-CTRL/lmc-exercises/blob/main/06_control_barrier_function.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Control Barrier Functions
In this exercise, you will familiarize yourself with control barrier functions, and their application as a safety filter.

In [ ]:
import abc
from typing import Callable
import jax.numpy as jnp
import matplotlib.pyplot as plt
import functools
import jax

import cvxpy as cp
from ipywidgets import interact

In [ ]:
# some helpers for plotting halfspaces
def _plot_halfspace_lessthan(
    normal_vector, constant, xlim=(-10, 10), ylim=(-10, 10), linestyle="-", alpha=0.5
):
    # Define the normal vector and constant
    a, b = normal_vector
    c = constant

    # Create a grid of points
    x = jnp.linspace(xlim[0], xlim[1], 400)
    y = jnp.linspace(ylim[0], ylim[1], 400)
    X, Y = jnp.meshgrid(x, y)

    # Calculate the values of the halfspace
    Z = a * X + b * Y + c

    # Plot the halfspace
    plt.contourf(X, Y, Z <= 0, alpha=[1.0, alpha], colors=["#ffb09c", "#E0FFD2"])

    plt.contour(X, Y, Z, levels=[0], colors="black", linestyles=linestyle)


    # Set the limits and labels
    plt.xlim(xlim)
    plt.ylim(ylim)
    plt.xlabel("x")
    plt.ylabel("y")

    plt.grid(True)
    plt.axhline(0, color="black", linewidth=0.5)
    plt.axvline(0, color="black", linewidth=0.5)
    # plt.show()


def plot_halfspace(normal_vector, constant, relation, xlim=(-10, 10), ylim=(-10, 10), alpha=0.5):
    if relation == "<=":
        _plot_halfspace_lessthan(
            normal_vector, constant, xlim=xlim, ylim=ylim, linestyle="-", alpha=alpha
        )
    elif relation == "<":
        _plot_halfspace_lessthan(
            normal_vector, constant, xlim=xlim, ylim=ylim, linestyle="--", alpha=alpha
        )
    elif relation == ">=":
        _plot_halfspace_lessthan(
            [-normal_vector[0], -normal_vector[1]],
            -constant,
            xlim=xlim,
            ylim=ylim,
            linestyle="-",
            alpha=alpha
        )
    elif relation == ">":
        _plot_halfspace_lessthan(
            [-normal_vector[0], -normal_vector[1]],
            -constant,
            xlim=xlim,
            ylim=ylim,
            linestyle="--",
            alpha=alpha
        )

def plot_box_constraint(lower, upper, xlim=(-10, 10), ylim=(-10, 10), alpha=0.5):
    # left
    plot_halfspace(
        [1, 0], -lower[0], ">=", xlim=xlim, ylim=ylim, alpha=alpha)
    # right
    plot_halfspace(
        [1, 0], -upper[0], "<=", xlim=xlim, ylim=ylim, alpha=alpha)
    # bottom
    plot_halfspace(
        [0, 1], -lower[1], ">=", xlim=xlim, ylim=ylim, alpha=alpha)
    # top
    plot_halfspace(
        [0, 1], -upper[1], "<=", xlim=xlim, ylim=ylim, alpha=alpha)


### Define a simple control-affine dynamics class


In [ ]:
class ControlAffineDynamics(metaclass=abc.ABCMeta):
    """Abstract base class for dynamical systems."""

    drift_dynamics: Callable[[jnp.ndarray, float], jnp.ndarray]
    control_matrix: Callable[[jnp.ndarray, float], jnp.ndarray]

    state_dim: int
    control_dim: int

    def __init__(
        self,
        drift_dynamics: Callable[[jnp.ndarray, float], jnp.ndarray],
        control_matrix: Callable[[jnp.ndarray, float], jnp.ndarray],
        state_dim: int,
        control_dim: int,
    ):
        """Initializes the Dynamics object.

        Args:
            dynamics_func: A callable representing the dynamics function.
            state_dim: The dimension of the state space.
            control_dim: The dimension of the control space.
        """
        self.drift_dynamics = drift_dynamics
        self.control_matrix = control_matrix
        self.state_dim = state_dim
        self.control_dim = control_dim

    def __call__(
        self, state: jnp.ndarray, control: jnp.ndarray, time: float = 0.0
    ) -> jnp.ndarray:
        """Evaluates the dynamics function at a given state, control, and time.

        Args:
            state: The current state.
            control: The current control input.
            time: The current time (optional, defaults to 0).

        Returns:
            The next state.
        """
        return (
            self.drift_dynamics(state, time)
            + self.control_matrix(state, time) @ control
        )

# define a simple 2D single integrator dynamics
def SingleIntegrator2D() -> ControlAffineDynamics:
    """Creates a single integrator dynamics object.

    Returns:
        A ControlAffineDynamics object representing the single integrator dynamics.
    """

    def drift_dynamics(state: jnp.ndarray, time: float = 0.0) -> jnp.ndarray:
        return jnp.zeros_like(state)

    def control_matrix(state: jnp.ndarray, time: float = 0.0) -> jnp.ndarray:
        return jnp.eye(state.shape[0])

    return ControlAffineDynamics(
        drift_dynamics, control_matrix, state_dim=2, control_dim=2
    )

In [ ]:
dynamics = SingleIntegrator2D()

### Set up functions related to CBFs and control constraints

In [ ]:
# define a simple control barrier function for a circular safe set
def control_barrier_function(state: jnp.ndarray) -> float:
    """Control barrier function for a circular safe set.

    Args:
        state: The current state.
    Returns:
        The value of the control barrier function at the given state.
    """
    # Define the center and radius of the circular safe set
    center = jnp.array([0.0, 0.0])
    radius = 2.0

    # Compute the distance from the center
    distance_squared = jnp.linalg.norm(state - center) ** 2

    # The control barrier function is positive inside the circle and negative outside
    # h(x) = r^2 - ||x - c||^2
    return radius**2 - distance_squared

# define the CBF constraint function
# also the same for CLF constraints
def cbf_constraint(
    cbf: Callable[[jnp.ndarray], float],
    dynamics: ControlAffineDynamics,
    state: jnp.ndarray,
    alpha: Callable[[float], float] = lambda h: h,
) -> float:
    """Computes the linear constraint for a control barrier function at a given state.

    Args:
        cbf: The control barrier function.
        dynamics: The control-affine dynamics of the system.
        state: The current state.
        alpha: An extended class K function (default is identity).

    Returns:
        A tuple (Lg_b, Lf_b + alpha(b(x))) representing the linear constraint coefficients.
    """
    # b(x)
    barrier_value = cbf(state)
    # \nabla b(x)
    db = jax.grad(cbf)(state)
    # \nabla b(x) f(x) + \nabla b(x) g(x) u + alpha(b(x)) >= 0
    Lf_b = db @ dynamics.drift_dynamics(state)
    Lg_b = db @ dynamics.control_matrix(state)
    return Lg_b, Lf_b + alpha(barrier_value)

### visualize CBF and constraints for a give state and desired control

In [ ]:
# set some example parameters to visualize the CBF and CBF constraint
a = 0.5
state = jnp.array([1.5, 0.5])
desired_control = jnp.array([1.0, 1.0])
linear, constant = cbf_constraint(
    cbf=control_barrier_function, dynamics=dynamics, state=state, alpha=lambda x: a * x
)

In [ ]:
# visualize the CBF constraint
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
X, Y = jnp.meshgrid(jnp.linspace(-3, 3, 100), jnp.linspace(-3, 3, 100))
XYs = jnp.stack([X, Y], axis=-1).reshape(-1, 2)
Z = jax.vmap(control_barrier_function)(XYs).reshape(X.shape)
plt.contourf(X, Y, Z, levels=20)
plt.colorbar(label="Control Barrier Function Value")
plt.contour(X, Y, Z, levels=[0], colors="black", linewidths=2)
plt.scatter(
    state[0], state[1], color="red", label="Current State", zorder=5
)
plt.arrow(
    state[0],
    state[1],
    desired_control[0] * 0.5,
    desired_control[1] * 0.5,
    head_width=0.1,
    head_length=0.15,
    fc="blue",
    ec="blue",
    length_includes_head=True,
    label="Desired Control Direction",
)
plt.title("Control Barrier Function")
plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")


plt.subplot(1, 2, 2)
xlim = (-2, 2)
ylim = (-2, 2)
lower = jnp.array([-1.0, -1.0])
upper = jnp.array([1.0, 1.0])
plot_halfspace(linear, constant, relation=">=", xlim=xlim, ylim=ylim, alpha=0.8)
plot_box_constraint(lower, upper, xlim=xlim, ylim=ylim, alpha=0.)
plt.scatter(
    desired_control[0],
    desired_control[1],
    color="orange",
    label="Desired Control",
    zorder=5,
    edgecolors="black",
    s=100,
)
plt.title("CBF Constraint on Control Input")

### Apply CBF safety filter

\begin{align}
\min_u & \: \| u - u_\mathrm{desired}\|^2_2\\
\mathrm{s.t.} & \:  \nabla b(x)^T [f(x) + B(x)u] + \alpha(b(x)) \geq 0\\
& \: u \leq u_{\max}\\
& \: u_{\min} \leq u
\end{align}

Use `cvxpy` to set up the optimization problem, and solve it

In [ ]:
# set some example parameters to visualize the CBF and CBF constraint
a = 0.05
state = jnp.array([1.5, 0.5])
desired_control = jnp.array([1.0, 1.0])
umin = -1.0
umax = 1.0
linear, constant = cbf_constraint(
    cbf=control_barrier_function, dynamics=dynamics, state=state, alpha=lambda x: a * x
)

In [ ]:
safe_control = cp.Variable(2)
objective = cp.Minimize(cp.sum_squares(safe_control - desired_control))
constraints = [
    linear @ safe_control + constant >= 0,
    safe_control <= umax,
    safe_control >= umin,
]
problem = cp.Problem(objective, constraints)
problem.solve()
print("safe control :", safe_control.value)


plt.figure(figsize=(5, 5))
plot_halfspace(linear, constant, relation=">=", xlim=xlim, ylim=ylim, alpha=0.8)
plot_box_constraint(lower, upper, xlim=xlim, ylim=ylim, alpha=0.)
plt.scatter(
    desired_control[0],
    desired_control[1],
    color="orange",
    label="Desired Control",
    zorder=5,
    edgecolors="black",
    s=100,
)
plt.scatter(
    safe_control.value[0],
    safe_control.value[1],
    color="yellow",
    label="Safe Control",
    zorder=5,
    edgecolors="black",
    s=100,
)
plt.legend()
plt.axis("equal")




### (i) What kind of operation is the optimization performing with respect to the feasible safe set?

[in-class discussion, student response here]

### (i) Weighted objective
Now, let's change the objective slightly,

\begin{align}
\min_u & \: (u - u_\mathrm{desired})^T Q (u - u_\mathrm{desired})\\
\mathrm{s.t.} & \:  \nabla b(x)^T [f(x) + B(x)u] + \alpha(b(x)) \geq 0\\
& \: u \leq u_{\max}\\
& \: u_{\min} \leq u
\end{align}

where $Q$ is a PSD matrix. Previously, $Q$ was just the identify matrix. Consider the case when is it not. For instance, what is the effect of choosing $Q = \mathrm{diag}([1.0, 2.0])$? Briefly explain.

[student response here]

In [ ]:
Q = jnp.diag(jnp.array([10., 1.0]))


In [ ]:
# Run code after, to help verify your response
# Run the optimization again with a different cost function
safe_control_Q = cp.Variable(2)
objective = cp.Minimize(cp.quad_form(safe_control_Q - desired_control, Q))
constraints = [
    linear @ safe_control_Q + constant >= 0,
    safe_control_Q <= 1.0,
    safe_control_Q >= -1.0,
]
problem = cp.Problem(objective, constraints)
problem.solve()
safe_control_Q.value




plt.figure(figsize=(5, 5))
plot_halfspace(linear, constant, relation=">=", xlim=xlim, ylim=ylim, alpha=0.8)
plot_box_constraint(lower, upper, xlim=xlim, ylim=ylim, alpha=0.)
plt.scatter(
    desired_control[0],
    desired_control[1],
    color="orange",
    label="Desired Control",
    zorder=5,
    edgecolors="black",
    s=100,
)
plt.scatter(
    safe_control.value[0],
    safe_control.value[1],
    color="yellow",
    label="Safe Control",
    zorder=5,
    edgecolors="black",
    s=100,
)

plt.scatter(
    safe_control_Q.value[0],
    safe_control_Q.value[1],
    color="forestgreen",
    label="Safe Control",
    zorder=5,
    edgecolors="black",
    s=100,
)
plt.legend()
plt.axis("equal")



## CBF-CLF-QP

Now, let's consider the case where we have both a CBF AND and CLF. The CBF is useful for ensuring the system stays within a safe set (e.g., avoid obstacle), and the CLF is useful for ensuring the system stablizes, or converges, to an equilibrium, or goal, state.
To do this, we can combine the CBF and CLF constraint. However, with these two constraints, there is a possibility that the resulting feasible set is *empty*. For example, moving away from the obstacle (satisfying CBF constraint) requires the system to also move away from the goal state (violating CLF constraint).
To ensure the problem remains feasible, we can introduce **slack variables** to relax some of the constraints by from margin $\epsilon$, but at the same time, require $\epsilon$ to be as small as possible to ensure the violation, if needed, minimal.


\begin{align}
\min_u & \: \| u - u_\mathrm{desired}\|^2_2 + \gamma_\mathrm{CBF} \|\epsilon_\mathrm{CBF}\|_2^2 + \gamma_\mathrm{CLF} \|\epsilon_\mathrm{CLF}\|_2^2\\
\mathrm{s.t.} & \:  \nabla b(x)^T [f(x) + B(x)u] + \alpha_\mathrm{CBF}(b(x)) \geq  - \epsilon_\mathrm{CBF}\\
& \:  \nabla V(x)^T [f(x) + B(x)u] + \alpha_\mathrm{CLF}(V(x)) \leq  \epsilon_\mathrm{CLF}\\
& \: u \leq u_{\max}\\
& \: u_{\min} \leq u\\
& \:  \epsilon_\mathrm{CBF} \geq 0, \: \epsilon_\mathrm{CLF} \geq 0
\end{align}



Consider a *reach-avoid* problem, where the system must reach a goal state, while avoiding a circular obstacle.

In [ ]:
# define a simple control barrier function for keeping outside a circular obstacle set
def control_barrier_function_obstacle(state: jnp.ndarray) -> float:
    """Control barrier function for keeping outside a circular obstacle set.

    Args:
        state: The current state.
    """
    pos = state[:2]
    # Define the center and radius of the circular obstacle set
    center = jnp.array([0.0, 0.0])
    radius = 2.0

    # Compute the distance from the center
    distance_squared = jnp.linalg.norm(pos - center) ** 2

    # The control barrier function is positive outside the obstacle set and negative inside
    return distance_squared - radius**2

def control_lyapunov_function_goal(state: jnp.ndarray, goal_pos: jnp.ndarray) -> float:
    """Control Lyapunov function for stabilizing to a goal position.

    Args:
        state: The current state.
        goal_pos: The goal position.
    """
    pos = state[:2]
    return jnp.linalg.norm(pos - goal_pos) ** 2


In [ ]:
# setting up parameters for both CBF and CLF constraints
# let alpha(x) = a * x for both CBF and CLF
cbf_a = 0.5
clf_a = 0.01
state = jnp.array([-4.0, 0.1])
goal_state = jnp.array([4.0, 0.0])
desired_control = jnp.array([2.0, 1.0])

umin = -3.0
umax = 3.0

# cbf constraint
cbf_linear_value, cbf_constant_value = cbf_constraint(
    cbf=control_barrier_function_obstacle, dynamics=dynamics, state=state, alpha=lambda x: cbf_a * x
)

# clf constraint
# NOTE: cbf_constraint function produces the same terms needed for the CLF constraint. Just input the CLF function instead of the CBF function.
clf_linear_value, clf_constant_value = cbf_constraint(
    cbf=functools.partial(control_lyapunov_function_goal, goal_pos=goal_state), dynamics=dynamics, state=state, alpha=lambda x: clf_a * x
)

In [ ]:
# visualize the CBF and CLF constraint
plt.figure(figsize=(20, 12))


plt.subplot(2, 3, 1)
xlim = jnp.abs(goal_state[0]) + 2
X, Y = jnp.meshgrid(jnp.linspace(-xlim, xlim, 100), jnp.linspace(-xlim, xlim, 100))
XYs = jnp.stack([X, Y], axis=-1).reshape(-1, 2)
Z = jax.vmap(control_barrier_function_obstacle)(XYs).reshape(X.shape)
plt.contourf(X, Y, Z, levels=20)
plt.colorbar(label="Control Barrier Function Value")
plt.contour(X, Y, Z, levels=[0], colors="black", linewidths=2)
plt.scatter(
    state[0], state[1], color="red", label="Current State", zorder=5
)
plt.scatter(
    goal_state[0], goal_state[1], color="C2", label="Goal State", zorder=5
)
plt.arrow(
    state[0],
    state[1],
    desired_control[0] * 0.5,
    desired_control[1] * 0.5,
    head_width=0.1,
    head_length=0.15,
    fc="yellow",
    ec="yellow",
    length_includes_head=True,
    label="Desired Control Direction",
)
plt.title("Control Barrier Function")
plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")
plt.legend()

plt.subplot(2, 3, 2)
xlim = jnp.abs(goal_state[0]) + 2
Z = jax.vmap(functools.partial(control_lyapunov_function_goal, goal_pos=goal_state))(XYs).reshape(X.shape)
plt.contourf(X, Y, Z, levels=20)
plt.colorbar(label="Control Lyapunov Function Value")
plt.contour(X, Y, Z, levels=[0], colors="black", linewidths=2)
plt.scatter(
    state[0], state[1], color="red", label="Current State", zorder=5
)
plt.scatter(
    goal_state[0], goal_state[1], color="C2", label="Goal State", zorder=5
)
plt.arrow(
    state[0],
    state[1],
    desired_control[0] * 0.5,
    desired_control[1] * 0.5,
    head_width=0.1,
    head_length=0.15,
    fc="yellow",
    ec="yellow",
    length_includes_head=True,
    label="Desired Control Direction",
)
plt.title("Control Lyapunov Function")
plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")
plt.legend()


# plotting control constraints
xlim = (umin-2, umax+2)
ylim = (umin-2, umax+2)
lower = jnp.array([umin, umin])
upper = jnp.array([umax, umax])

plt.subplot(2, 3, 3)
plot_halfspace(cbf_linear_value, cbf_constant_value, relation=">=", xlim=xlim, ylim=ylim, alpha=0.8)
plot_halfspace(clf_linear_value, clf_constant_value, relation="<=", xlim=xlim, ylim=ylim, alpha=0.3)

plot_box_constraint(lower, upper, xlim=xlim, ylim=ylim, alpha=0.)
plt.scatter(
    desired_control[0],
    desired_control[1],
    color="yellow",
    label="Desired Control",
    zorder=5,
    s=100,
    edgecolors="black",
)
plt.title("CBF - CLF Constraint on Control Input")

plt.subplot(2, 3, 4)
plot_halfspace(cbf_linear_value, cbf_constant_value, relation=">=", xlim=xlim, ylim=ylim, alpha=0.8)
plot_box_constraint(lower, upper, xlim=xlim, ylim=ylim, alpha=0.)
plt.scatter(
    desired_control[0],
    desired_control[1],
    color="yellow",
    label="Desired Control",
    zorder=5,
    s=100,
    edgecolors="black",
)
plt.title("CBF Constraint on Control Input")

plt.subplot(2, 3, 5)
plot_halfspace(clf_linear_value, clf_constant_value, relation="<=", xlim=xlim, ylim=ylim, alpha=0.3)
plot_box_constraint(lower, upper, xlim=xlim, ylim=ylim, alpha=0.)
plt.scatter(
    desired_control[0],
    desired_control[1],
    color="yellow",
    label="Desired Control",
    zorder=5,
    s=100,
    edgecolors="black",
)
plt.title("CLF Constraint on Control Input")

### (iii) What is the effect of `cbf_a` and `clf_a`?

In [ ]:
# set up the CVXPY problem with both CBF and CLF constraints

safe_control = cp.Variable(2, name="safe_control")
cbf_epsilon = cp.Variable(1, name="cbf_epsilon")
clf_epsilon = cp.Variable(1, name="clf_epsilon")


cbf_linear = cp.Parameter(2, "cbf_linear")
cbf_constant = cp.Parameter(1, "cbf_constant")
clf_linear = cp.Parameter(2, "clf_linear")
clf_constant = cp.Parameter(1, "clf_constant")
cbf_gamma = cp.Constant(1, "cbf_gamma")
clf_gamma = cp.Constant(1, "clf_gamma")

objective = cp.Minimize(
    cp.sum_squares(safe_control - desired_control)
    + cbf_gamma * cp.sum_squares(cbf_epsilon)
    + clf_gamma * cp.sum_squares(clf_epsilon)
)

constraints = [
    cbf_linear @ safe_control + cbf_constant >= -cbf_epsilon,
    clf_linear @ safe_control + clf_constant <= clf_epsilon,
    safe_control <= umax,
    safe_control >= umin,
    cbf_epsilon >= 0.0,
    clf_epsilon >= 0.0,
]

problem = cp.Problem(objective, constraints)

In [ ]:

# cbf constraint
cbf_linear_value, cbf_constant_value = cbf_constraint(
    cbf=control_barrier_function_obstacle, dynamics=dynamics, state=state, alpha=lambda x: cbf_a * x
)

# clf constraint
# NOTE: cbf_constraint function produces the same terms needed for the CLF constraint. Just input the CLF function instead of the CBF function.
clf_linear_value, clf_constant_value = cbf_constraint(
    cbf=functools.partial(control_lyapunov_function_goal, goal_pos=goal_state), dynamics=dynamics, state=state, alpha=lambda x: clf_a * x
)

In [ ]:
# write function to solve the QP problem given the current state
def solve_cbf_clf_qp(problem, state, dynamics, cbf, clf, goal_state, alpha_cbf, alpha_clf):
    # cbf constraint
    cbf_linear_value, cbf_constant_value = cbf_constraint(
        cbf=cbf, dynamics=dynamics, state=state, alpha=alpha_cbf
    )
    clf_linear_value, clf_constant_value = cbf_constraint(
        cbf=functools.partial(clf, goal_pos=goal_state), dynamics=dynamics, state=state, alpha=alpha_clf
    )
    cbf_linear.project_and_assign(cbf_linear_value)
    cbf_constant.project_and_assign(cbf_constant_value)
    clf_linear.project_and_assign(clf_linear_value)
    clf_constant.project_and_assign(clf_constant_value)
    problem.solve()
    return problem.var_dict["safe_control"].value


# setting up parameters for both CBF and CLF constraints
# let alpha(x) = a * x for both CBF and CLF
cbf_a = 1.0
clf_a = 0.1
goal_state = jnp.array([4.0, 0.0])

policy = functools.partial(
    solve_cbf_clf_qp,
    problem,
    dynamics=dynamics,
    cbf=control_barrier_function_obstacle,
    clf=control_lyapunov_function_goal,
    goal_state=goal_state,
    alpha_cbf=lambda x: cbf_a * x,
    alpha_clf=lambda x: clf_a * x,
)

In [ ]:
# setting up the episode
initial_state = jnp.array([-4.0, 0.1])
goal_state = jnp.array([4.0, 0.0])
desired_control = jnp.array([1.0, 0.0])

# set the gamma values
cbf_gamma.project_and_assign(1000.0)
clf_gamma.project_and_assign(10.0)

tmax = 10.0
dt = 0.05
xs = [initial_state]
us = []
# run the episode
for t in jnp.arange(0, tmax, dt):
    state = xs[-1]
    u = policy(state)
    next_state = state + dynamics(state, u) * dt
    xs.append(next_state)
    us.append(u)
    if jnp.linalg.norm(next_state - goal_state) < 0.05:
        print(f"Reached goal at time {t:.2f}s")
        break
xs = jnp.stack(xs)
us = jnp.stack(us)

In [ ]:
# visualize the CBF and CLF constraint
@interact(t=(0, xs.shape[0]-1, 1))
def plot_cbf_clf_trajectory(t):
    state = xs[t]
    control = us[t]
    plt.figure(figsize=(20, 12))

    # plot trajectory overlayed on CBF contour
    plt.subplot(2, 3, 1)
    xlim = (xs[:,0].min()-2, xs[:,0].max()+2)
    X, Y = jnp.meshgrid(jnp.linspace(*xlim, 100), jnp.linspace(*xlim, 100))
    XYs = jnp.stack([X, Y], axis=-1).reshape(-1, 2)
    Z = jax.vmap(control_barrier_function_obstacle)(XYs).reshape(X.shape)
    plt.contourf(X, Y, Z, levels=20)
    plt.colorbar(label="Control Barrier Function Value")
    plt.contour(X, Y, Z, levels=[0], colors="black", linewidths=2)
    plt.plot(xs[:, 0], xs[:, 1], color="lightskyblue", linewidth=2, label="Trajectory")
    plt.scatter(
        state[0], state[1], color="red", label="Current State", zorder=5
    )
    plt.scatter(
        goal_state[0], goal_state[1], color="C2", label="Goal State", zorder=5
    )
    plt.arrow(
        state[0],
        state[1],
        control[0],
        control[1],
        head_width=0.1,
        head_length=0.15,
        fc="orange",
        ec="orange",
        length_includes_head=True,
        label="Actual Control Direction",
        zorder=10
    )

    plt.arrow(
        state[0],
        state[1],
        desired_control[0],
        desired_control[1],
        head_width=0.1,
        head_length=0.15,
        fc="yellow",
        ec="yellow",
        length_includes_head=True,
        label="Desired Control Direction",
        zorder=10
    )
    plt.title("Control Barrier Function")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.axis("equal")
    plt.legend(loc="lower left")

    # plot trajectory overlayed on CLF contour
    plt.subplot(2, 3, 2)
    xlim = jnp.abs(goal_state[0]) + 2
    Z = jax.vmap(functools.partial(control_lyapunov_function_goal, goal_pos=goal_state))(XYs).reshape(X.shape)
    plt.contourf(X, Y, Z, levels=20)
    plt.colorbar(label="Control Lyapunov Function Value")
    plt.contour(X, Y, Z, levels=[0], colors="black", linewidths=2)
    plt.plot(xs[:, 0], xs[:, 1], color="lightskyblue", linewidth=2, label="Trajectory")
    plt.scatter(
        state[0], state[1], color="red", label="Current State", zorder=5
    )
    plt.scatter(
        goal_state[0], goal_state[1], color="C2", label="Goal State", zorder=5
    )
    plt.arrow(
        state[0],
        state[1],
        control[0],
        control[1],
        head_width=0.1,
        head_length=0.15,
        fc="orange",
        ec="orange",
        length_includes_head=True,
        label="Actual Control Direction",
        zorder=10,
    )

    plt.arrow(
        state[0],
        state[1],
        desired_control[0],
        desired_control[1],
        head_width=0.1,
        head_length=0.15,
        fc="yellow",
        ec="yellow",
        length_includes_head=True,
        label="Desired Control Direction",
        zorder=10,
    )
    plt.title("Control Lyapunov Function")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.axis("equal")
    plt.legend(loc="lower left")


    # combined CBF and CLF control constraints
    plt.subplot(2, 3, 3)
    xlim = (umin-2, umax+2)
    ylim = (umin-2, umax+2)
    lower = jnp.array([umin, umin])
    upper = jnp.array([umax, umax])

    # cbf constraint
    cbf_linear_value, cbf_constant_value = cbf_constraint(
        cbf=control_barrier_function_obstacle, dynamics=dynamics, state=state, alpha=lambda x: cbf_a * x
    )

    # clf constraint
    # NOTE: cbf_constraint function produces the same terms needed for the CLF constraint. Just input the CLF function instead of the CBF function.
    clf_linear_value, clf_constant_value = cbf_constraint(
        cbf=functools.partial(control_lyapunov_function_goal, goal_pos=goal_state), dynamics=dynamics, state=state, alpha=lambda x: clf_a * x
    )

    plot_halfspace(cbf_linear_value, cbf_constant_value, relation=">=", xlim=xlim, ylim=ylim, alpha=0.)
    plot_halfspace(clf_linear_value, clf_constant_value, relation="<=", xlim=xlim, ylim=ylim, alpha=0.3)

    plot_box_constraint(lower, upper, xlim=xlim, ylim=ylim, alpha=0.)

    plt.scatter(
        control[0],
        control[1],
        color="orange",
        label="Actual Control",
        zorder=5,
        s=100,
        edgecolors="black",
    )
    plt.scatter(
        desired_control[0],
        desired_control[1],
        color="yellow",
        label="Desired Control",
        zorder=5,
        s=100,
        edgecolors="black",
    )
    plt.title("CBF - CLF Constraint on Control Input")

    # CBF value over time
    plt.subplot(2, 3, 4)
    cbf_value = jax.vmap(control_barrier_function_obstacle)(xs)
    plt.plot(jnp.arange(xs.shape[0]) * dt, cbf_value, label="CBF Value")
    plt.xlabel("Time (s)")
    plt.ylabel("CBF Value")
    plt.title("CBF Value Over Time")
    plt.hlines(0.0, 0, xs.shape[0]*dt, colors="red", linestyles="--", label="Boundary")
    plt.grid(alpha=0.5)
    plt.legend()

    # CLF value over time
    plt.subplot(2, 3, 5)
    clf_value = jax.vmap(functools.partial(control_lyapunov_function_goal, goal_pos=goal_state))(xs)
    plt.plot(jnp.arange(xs.shape[0]) * dt, clf_value, label="CLF Value")
    plt.hlines(0.0, 0, xs.shape[0]*dt, colors="red", linestyles="--", label="Boundary")
    plt.xlabel("Time (s)")
    plt.ylabel("CLF Value")
    plt.title("CLF Value Over Time")
    plt.grid(alpha=0.5)
    plt.legend()

    # control inputs over time
    plt.subplot(2,3,6)
    plt.plot(jnp.arange(us.shape[0]) * dt, us[:, 0], label="u1")
    plt.plot(jnp.arange(us.shape[0]) * dt, us[:, 1], label="u2")
    plt.axhline(umax, color="red", linestyle="--", label="Control Limits")
    plt.axhline(umin, color="red", linestyle="--")
    plt.grid(alpha=0.5)
    plt.xlabel("Time (s)")
    plt.ylabel("Control Input")
    plt.legend()
    plt.title("Control Inputs Over Time")
    plt.tight_layout()
    plt.show()
